# Thong ke class 0/1/2 cho 40 set train_val

Notebook nay quet 4 ratio (`1_1_1`, `2_1_1`, `4_1_1`, `9_1_1`) x 10 set = 40 set,
va thong ke so mau cho tung lop 0/1/2 tren ca train va val.

In [1]:
from pathlib import Path
import pandas as pd
import torch

DATA_ROOT = Path(r"D:\\Bio_sequence_Research_AITALAB\\train\\task1_splicing_prediction\\data_preparation\\train_val")
RATIOS = ["1_1_1", "2_1_1", "4_1_1", "9_1_1"]
EXPECTED_SETS_PER_RATIO = 10
LABEL_COL = "Splicing_types"

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Khong tim thay thu muc: {DATA_ROOT}")

DATA_ROOT

WindowsPath('D:/Bio_sequence_Research_AITALAB/train/task1_splicing_prediction/data_preparation/train_val')

In [2]:
def _safe_label_counts(labels_series):
    """Tra ve dict count cho 0/1/2, key nao thieu thi tra 0."""
    counts = labels_series.value_counts().to_dict()
    return {
        0: int(counts.get(0, 0)),
        1: int(counts.get(1, 0)),
        2: int(counts.get(2, 0)),
    }


def read_counts_from_pt(pt_path):
    data = torch.load(pt_path, map_location="cpu")
    if not isinstance(data, dict) or "labels" not in data:
        raise ValueError(f"File .pt khong dung dinh dang mong doi: {pt_path}")

    labels = data["labels"]
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy().tolist()

    labels_series = pd.Series(labels, dtype="int64")
    counts = _safe_label_counts(labels_series)

    return {
        "total": int(len(labels_series)),
        "class_0": counts[0],
        "class_1": counts[1],
        "class_2": counts[2],
        "file_type": "pt",
    }


def read_counts_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    if LABEL_COL not in df.columns:
        raise ValueError(f"CSV thieu cot nhan '{LABEL_COL}': {csv_path}")

    labels_series = pd.to_numeric(df[LABEL_COL], errors="coerce").dropna().astype(int)
    counts = _safe_label_counts(labels_series)

    return {
        "total": int(len(labels_series)),
        "class_0": counts[0],
        "class_1": counts[1],
        "class_2": counts[2],
        "file_type": "csv",
    }


def get_split_stats(set_dir, split_name):
    """Uu tien doc .pt; neu khong co thi doc .csv."""
    pt_file = set_dir / f"{split_name}_embeddings.pt"
    csv_file = set_dir / f"{split_name}.csv"

    if pt_file.exists():
        stats = read_counts_from_pt(pt_file)
        stats["source_path"] = str(pt_file)
        return stats

    if csv_file.exists():
        stats = read_counts_from_csv(csv_file)
        stats["source_path"] = str(csv_file)
        return stats

    raise FileNotFoundError(
        f"Khong tim thay file cho split '{split_name}' trong {set_dir}.\n"
        f"Can co mot trong hai file: {pt_file.name} hoac {csv_file.name}"
    )


rows = []
for ratio in RATIOS:
    ratio_dir = DATA_ROOT / ratio
    if not ratio_dir.exists():
        raise FileNotFoundError(f"Thieu thu muc ratio: {ratio_dir}")

    for set_idx in range(1, EXPECTED_SETS_PER_RATIO + 1):
        set_dir = ratio_dir / f"set_{set_idx}"
        if not set_dir.exists():
            raise FileNotFoundError(f"Thieu thu muc set: {set_dir}")

        for split in ["train", "val"]:
            stats = get_split_stats(set_dir, split)
            rows.append({
                "ratio": ratio,
                "set": set_idx,
                "split": split,
                **stats,
            })

stats_df = pd.DataFrame(rows).sort_values(["ratio", "set", "split"]).reset_index(drop=True)

print(f"So dong thong ke (du kien 80 = 40 set x train/val): {len(stats_df)}")
stats_df.head(12)

So dong thong ke (du kien 80 = 40 set x train/val): 80


,ratio,set,split,total,class_0,class_1,class_2,file_type,source_path
0,1_1_1,1,train,612931,205583,205620,201728,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
1,1_1_1,1,val,108165,36425,36388,35352,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
2,1_1_1,2,train,612931,205873,205593,201465,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
3,1_1_1,2,val,108165,36135,36415,35615,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
4,1_1_1,3,train,612931,205878,205678,201375,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
5,1_1_1,3,val,108165,36130,36330,35705,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
6,1_1_1,4,train,612931,205501,205635,201795,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
7,1_1_1,4,val,108165,36507,36373,35285,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
8,1_1_1,5,train,612931,205584,205931,201416,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...
9,1_1_1,5,val,108165,36424,36077,35664,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...


In [3]:
# Tong hop theo ratio va split
summary_ratio_split = (
    stats_df
    .groupby(["ratio", "split"], as_index=False)[["total", "class_0", "class_1", "class_2"]]
    .sum()
    .sort_values(["ratio", "split"])
)

summary_ratio_split

,ratio,split,total,class_0,class_1,class_2
0,1_1_1,train,6129310,2056258,2057369,2015683
1,1_1_1,val,1081650,363822,362711,355117
2,2_1_1,train,8186380,4114967,2056502,2014911
3,2_1_1,val,1444660,725193,363578,355889
4,4_1_1,train,12300520,8227732,2056913,2015875
5,4_1_1,val,2170680,1452588,363167,354925
6,9_1_1,train,22585860,18513358,2057618,2014884
7,9_1_1,val,3985740,3267362,362462,355916


In [4]:
# Kiem tra nhanh so set moi ratio
set_count_check = (
    stats_df[["ratio", "set"]]
    .drop_duplicates()
    .groupby("ratio", as_index=False)
    .size()
    .rename(columns={"size": "num_sets"})
)

set_count_check

,ratio,num_sets
0,1_1_1,10
1,2_1_1,10
2,4_1_1,10
3,9_1_1,10


In [5]:
# Luu ket qua ra CSV de dung lai
out_dir = DATA_ROOT.parent / "statistics"
out_dir.mkdir(parents=True, exist_ok=True)

detail_csv = out_dir / "train_val_40sets_class_distribution_detail.csv"
summary_csv = out_dir / "train_val_40sets_class_distribution_summary_ratio_split.csv"

stats_df.to_csv(detail_csv, index=False)
summary_ratio_split.to_csv(summary_csv, index=False)

print(f"Da luu file chi tiet: {detail_csv}")
print(f"Da luu file tong hop: {summary_csv}")

Da luu file chi tiet: D:\Bio_sequence_Research_AITALAB\train\task1_splicing_prediction\data_preparation\statistics\train_val_40sets_class_distribution_detail.csv
Da luu file tong hop: D:\Bio_sequence_Research_AITALAB\train\task1_splicing_prediction\data_preparation\statistics\train_val_40sets_class_distribution_summary_ratio_split.csv


In [8]:
# ===== GOP TRAIN + VAL THANH TRAIN_VAL =====
# stats_df hien tai dang co 2 dong moi set: split=train va split=val
train_val_merged_df = (
    stats_df
    .groupby(["ratio", "set"], as_index=False)[["total", "class_0", "class_1", "class_2"]]
    .sum()
    .sort_values(["ratio", "set"])
    .reset_index(drop=True)
)

# Neu muon giu nhan split de de nhin
train_val_merged_df["split"] = "train_val"

print(f"So dong train_val sau khi gop (du kien 40): {len(train_val_merged_df)}")
train_val_merged_df.head(12)

So dong train_val sau khi gop (du kien 40): 40


,ratio,set,total,class_0,class_1,class_2,split
0,1_1_1,1,721096,242008,242008,237080,train_val
1,1_1_1,2,721096,242008,242008,237080,train_val
2,1_1_1,3,721096,242008,242008,237080,train_val
3,1_1_1,4,721096,242008,242008,237080,train_val
4,1_1_1,5,721096,242008,242008,237080,train_val
5,1_1_1,6,721096,242008,242008,237080,train_val
6,1_1_1,7,721096,242008,242008,237080,train_val
7,1_1_1,8,721096,242008,242008,237080,train_val
8,1_1_1,9,721096,242008,242008,237080,train_val
9,1_1_1,10,721096,242008,242008,237080,train_val


In [9]:
# ===== TONG HOP THEO RATIO (DA GOP TRAIN_VAL) =====
summary_ratio_train_val = (
    train_val_merged_df
    .groupby(["ratio"], as_index=False)[["total", "class_0", "class_1", "class_2"]]
    .sum()
    .sort_values(["ratio"])
)

summary_ratio_train_val

,ratio,total,class_0,class_1,class_2
0,1_1_1,7210960,2420080,2420080,2370800
1,2_1_1,9631040,4840160,2420080,2370800
2,4_1_1,14471200,9680320,2420080,2370800
3,9_1_1,26571600,21780720,2420080,2370800


In [10]:
# ===== KIEM TRA SO SET MOI RATIO =====
set_count_check = (
    train_val_merged_df[["ratio", "set"]]
    .drop_duplicates()
    .groupby("ratio", as_index=False)
    .size()
    .rename(columns={"size": "num_sets"})
)

set_count_check

,ratio,num_sets
0,1_1_1,10
1,2_1_1,10
2,4_1_1,10
3,9_1_1,10


In [11]:
# ===== SAVE TRAIN_VAL (DA GOP) + TEST CSV =====
out_dir = DATA_ROOT.parent / "statistics"
out_dir.mkdir(parents=True, exist_ok=True)

train_val_detail_csv = out_dir / "train_val_40sets_class_distribution_detail_merged.csv"
train_val_summary_csv = out_dir / "train_val_40sets_class_distribution_summary_ratio.csv"
test_detail_csv = out_dir / "test_gencode_gtex_class_distribution_detail.csv"

train_val_merged_df.to_csv(train_val_detail_csv, index=False)
summary_ratio_train_val.to_csv(train_val_summary_csv, index=False)

print(f"Da luu train_val chi tiet (gop): {train_val_detail_csv}")
print(f"Da luu train_val tong hop: {train_val_summary_csv}")

Da luu train_val chi tiet (gop): D:\Bio_sequence_Research_AITALAB\train\task1_splicing_prediction\data_preparation\statistics\train_val_40sets_class_distribution_detail_merged.csv
Da luu train_val tong hop: D:\Bio_sequence_Research_AITALAB\train\task1_splicing_prediction\data_preparation\statistics\train_val_40sets_class_distribution_summary_ratio.csv


In [6]:
# ===== TEST STATS (gencode + gtex) =====
# Yeu cau:
# - ratio 1_1_1, 2_1_1, 4_1_1, 9_1_1
# - ratio 100_1_1 dung test_data / gtex_test_data
# - fallback neu repo dat ten 10_1_1 thay vi 9_1_1

TEST_RATIO_SPECS = [
    {"ratio": "1_1_1",   "gencode_candidates": ["test_1_1_1"],               "gtex_candidates": ["gtex_test_1_1_1"]},
    {"ratio": "2_1_1",   "gencode_candidates": ["test_2_1_1"],               "gtex_candidates": ["gtex_test_2_1_1"]},
    {"ratio": "4_1_1",   "gencode_candidates": ["test_4_1_1"],               "gtex_candidates": ["gtex_test_4_1_1"]},
    {"ratio": "9_1_1",   "gencode_candidates": ["test_9_1_1", "test_10_1_1"], "gtex_candidates": ["gtex_test_9_1_1", "gtex_test_10_1_1"]},
    {"ratio": "100_1_1", "gencode_candidates": ["test_data"],                "gtex_candidates": ["gtex_test_data"]},
]

def get_stats_from_stem(parent_dir, stem):
    pt_file = parent_dir / f"{stem}_embeddings.pt"
    csv_file = parent_dir / f"{stem}.csv"

    if pt_file.exists():
        stats = read_counts_from_pt(pt_file)
        stats["source_path"] = str(pt_file)
        stats["file_stem"] = stem
        return stats

    if csv_file.exists():
        stats = read_counts_from_csv(csv_file)
        stats["source_path"] = str(csv_file)
        stats["file_stem"] = stem
        return stats

    return None

def resolve_from_candidates(parent_dir, candidates, dataset_kind, ratio):
    for stem in candidates:
        stats = get_stats_from_stem(parent_dir, stem)
        if stats is not None:
            return {
                "dataset_kind": dataset_kind,   # test_gencode | test_gtex
                "ratio": ratio,
                **stats
            }

    raise FileNotFoundError(
        f"Khong tim thay file cho {dataset_kind}, ratio={ratio}. "
        f"Da thu: {candidates}"
    )

test_rows = []
for spec in TEST_RATIO_SPECS:
    ratio = spec["ratio"]

    test_rows.append(
        resolve_from_candidates(
            DATA_ROOT,
            spec["gencode_candidates"],
            dataset_kind="test_gencode",
            ratio=ratio
        )
    )

    test_rows.append(
        resolve_from_candidates(
            DATA_ROOT,
            spec["gtex_candidates"],
            dataset_kind="test_gtex",
            ratio=ratio
        )
    )

test_df = (
    pd.DataFrame(test_rows)
    .sort_values(["dataset_kind", "ratio"])
    .reset_index(drop=True)
)

print(f"So dong test (du kien 10): {len(test_df)}")
test_df

So dong test (du kien 10): 10


,dataset_kind,ratio,total,class_0,class_1,class_2,file_type,source_path,file_stem
0,test_gencode,100_1_1,938297,920809,8822,8666,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,test_data
1,test_gencode,1_1_1,26310,8822,8822,8666,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,test_1_1_1
2,test_gencode,2_1_1,35132,17644,8822,8666,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,test_2_1_1
3,test_gencode,4_1_1,52776,35288,8822,8666,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,test_4_1_1
4,test_gencode,9_1_1,105708,88220,8822,8666,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,test_10_1_1
5,test_gtex,100_1_1,1402857,1375350,13776,13731,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,gtex_test_data
6,test_gtex,1_1_1,41283,13776,13776,13731,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,gtex_test_1_1_1
7,test_gtex,2_1_1,55059,27552,13776,13731,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,gtex_test_2_1_1
8,test_gtex,4_1_1,82611,55104,13776,13731,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,gtex_test_4_1_1
9,test_gtex,9_1_1,165267,137760,13776,13731,pt,D:\Bio_sequence_Research_AITALAB\train\task1_s...,gtex_test_10_1_1


In [7]:
# ===== SAVE TEST STATS CSV =====
out_dir = DATA_ROOT.parent / "statistics"
out_dir.mkdir(parents=True, exist_ok=True)

test_detail_csv = out_dir / "test_gencode_gtex_class_distribution_detail.csv"
test_df.to_csv(test_detail_csv, index=False)

print(f"Da luu test chi tiet: {test_detail_csv}")

Da luu test chi tiet: D:\Bio_sequence_Research_AITALAB\train\task1_splicing_prediction\data_preparation\statistics\test_gencode_gtex_class_distribution_detail.csv
